In [4]:
# 🎓 Assignment-04: House Price Prediction (High Performance)
# Ali-Salad | Bootcamp Submission

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# -----------------------------
# 0. Debug: Confirm file exists
# -----------------------------
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir())

# -----------------------------
# 1. Load and Clean Dataset
# -----------------------------
df = pd.read_csv("clean_house_dataset.csv")

# Ensure Price is numeric (strip $ if any remain)
df["Price"] = df["Price"].replace('[\$,]', '', regex=True).astype(float)

# Fix Location typos if needed
df["Location"] = df["Location"].replace({"Subrb": "Suburb", "??": "Unknown"})
df["Location"] = df["Location"].fillna("Unknown")

# Handle missing numeric values
for col in ["Size_sqft", "Bedrooms", "Bathrooms", "YearBuilt"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Feature engineering
df["HouseAge"] = 2025 - df["YearBuilt"]
df["BedBath"] = df["Bedrooms"] * df["Bathrooms"]
df["SizePerBedroom"] = df["Size_sqft"] / (df["Bedrooms"].replace(0, np.nan))

# Log-transform target
df["LogPrice"] = np.log(df["Price"])

# -----------------------------
# 2. Prepare Features & Target
# -----------------------------
X = df.drop(columns=["Price", "LogPrice"])
y = df["LogPrice"]

# Convert categorical variables (Location) to dummy variables
X = pd.get_dummies(X, drop_first=True)

# -----------------------------
# 3. Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 4. Train Models
# -----------------------------
lr_model = LinearRegression()

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    random_state=42
)

gb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=10,
    random_state=42
)

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

# -----------------------------
# 5. Evaluation Helper Function
# -----------------------------
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    # back-transform predictions
    y_pred_exp = np.exp(y_pred)
    y_test_exp = np.exp(y_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test_exp, y_pred_exp)
    mse = mean_squared_error(y_test_exp, y_pred_exp)
    rmse = np.sqrt(mse)
    
    print(f"{name} Performance:")
    print(f"  R²   : {r2:.2f}")
    print(f"  MAE  : {mae:,.0f}")
    print(f"  RMSE : {rmse:,.0f}")
    print("-" * 40)

# -----------------------------
# 6. Evaluate All Models
# -----------------------------
evaluate_model(lr_model, X_test, y_test, "Linear Regression")
evaluate_model(rf_model, X_test, y_test, "Random Forest (Tuned)")
evaluate_model(gb_model, X_test, y_test, "Gradient Boosting")

# -----------------------------
# 7. Single-row Sanity Check
# -----------------------------
i = 5  # pick any index from test set
row = X_test.iloc[[i]]
actual_price = np.exp(y_test.iloc[i])

lr_pred = np.exp(lr_model.predict(row)[0])
rf_pred = np.exp(rf_model.predict(row)[0])
gb_pred = np.exp(gb_model.predict(row)[0])

print("Sanity Check (Row {})".format(i))
print(f"Actual Price       : {actual_price:,.0f}")
print(f"Linear Regression  : {lr_pred:,.0f}")
print(f"Random Forest      : {rf_pred:,.0f}")
print(f"Gradient Boosting  : {gb_pred:,.0f}")


Current working directory: C:\Users\Alisalad\OneDrive\Desktop\house-price pedic\dataset
Files here: ['.ipynb_checkpoints', 'clean_house_dataset.csv', 'house_price_prediction.ipynb']
Linear Regression Performance:
  R²   : 0.48
  MAE  : 203,161
  RMSE : 530,124
----------------------------------------
Random Forest (Tuned) Performance:
  R²   : 0.53
  MAE  : 173,368
  RMSE : 527,172
----------------------------------------
Gradient Boosting Performance:
  R²   : 0.53
  MAE  : 175,854
  RMSE : 520,681
----------------------------------------
Sanity Check (Row 5)
Actual Price       : 327,300
Linear Regression  : 386,043
Random Forest      : 382,481
Gradient Boosting  : 356,281
